# Assertions — PIT Big-Cap and NYSE P20–P50 Datasets

This notebook is read-only. It validates the two outputs of `notebooks/01_build_pit_big_small_caps.ipynb` against the canonical ordinary-common-stock source produced by notebook 00: primary keys, exact formation memberships, NYSE breakpoint construction, exchange scope, PIT timing, delisting exposure, and actual 60-month daily completeness.

In [ ]:
# 1. CONFIGURATION
from pathlib import Path
import polars as pl

ROOT = Path.cwd()
if ROOT.name in {'notebooks', 'tests'}:
    ROOT = ROOT.parent

SRC = ROOT / 'data' / 'processed' / 'crsp_daily_common_stock_pit_source-1990-2025.parquet'
BIG = ROOT / 'data' / 'processed' / 'nyse_big_caps_pit_daily.parquet'
SMALL = ROOT / 'data' / 'processed' / 'nyse_small_caps_p20_p50_pit_daily.parquet'
INVESTABLE_EXCHANGES = ['N', 'A', 'Q']
NYSE_CODE = 'N'
W_MAX = 60
TOP_N = 100

for path in (SRC, BIG, SMALL):
    assert path.exists(), f'Missing file: {path}'
print('All input files are present.')

## 2. Schema, primary key, and binary flags

In [ ]:
required = {
    'PERMNO', 'Ticker', 'PrimaryExch', 'DlyCalDt', 'DlyRet', 'DlyRetx', 'DlyRetI',
    'DlyPrc', 'DlyCap', 'active_month', 'formation_month', 'is_active',
    'bucket', 'rank', 'formation_mktcap', 'bp_p20', 'bp_p50', 'bp_p90',
    'strict_hist60', 'formation_member_month', 'formation_member_active_month',
    'is_formation_member', 'formation_member_exchange', 'formation_member_bucket',
    'formation_member_rank', 'formation_member_mktcap',
    'formation_member_bp_p20', 'formation_member_bp_p50',
    'formation_member_bp_p90', 'formation_member_n_nyse_snapshot',
    'formation_member_strict_hist60', 'is_delisting_event', 'DelRet_event',
    'is_common_stock_10_11', 'is_regular_active',
    'is_investable_exchange', 'is_formation_eligible',
}

datasets = {
    'big_caps': pl.scan_parquet(BIG),
    'small_caps_p20_p50': pl.scan_parquet(SMALL),
}

for name, frame in datasets.items():
    missing = required - set(frame.collect_schema().names())
    assert not missing, f'{name}: missing columns {sorted(missing)}'
    stats = frame.select(
        pl.len().alias('rows'),
        pl.struct(['PERMNO', 'DlyCalDt']).is_duplicated().sum().alias('duplicate_keys'),
        (~pl.col('is_active').is_in([0, 1])).sum().alias('bad_active_flag'),
        (~pl.col('is_formation_member').is_in([0, 1])).sum().alias('bad_formation_flag'),
        (
            (pl.col('is_formation_member') == 1)
            & ~(
                pl.col('is_common_stock_10_11')
                & pl.col('is_regular_active')
                & pl.col('is_investable_exchange')
                & pl.col('is_formation_eligible')
            )
        ).sum().alias('ineligible_formation_members'),
    ).collect(engine='streaming')
    assert stats['duplicate_keys'][0] == 0, f'{name}: duplicate primary keys'
    assert stats['bad_active_flag'][0] == 0, f'{name}: invalid is_active flag'
    assert stats['bad_formation_flag'][0] == 0, f'{name}: invalid is_formation_member flag'
    assert stats['ineligible_formation_members'][0] == 0, f'{name}: ineligible formation member'
    print(name)
    display(stats)

## 3. Exact formation membership and PIT lag

Formation membership is reconstructed only from `is_formation_member`. Every month must contain the complete rank set 1 through 100. Observed presence during $M+1$ is deliberately not used.

In [ ]:
def extract_formation_membership(frame: pl.LazyFrame) -> pl.DataFrame:
    return (
        frame.filter(pl.col('is_formation_member') == 1)
        .select([
            'DlyCalDt', 'PERMNO',
            pl.col('formation_member_month').alias('formation_month'),
            pl.col('formation_member_active_month').alias('active_month'),
            pl.col('formation_member_exchange').alias('formation_exchange'),
            pl.col('formation_member_bucket').alias('bucket'),
            pl.col('formation_member_rank').alias('rank'),
            pl.col('formation_member_mktcap').alias('formation_mktcap'),
            pl.col('formation_member_strict_hist60').alias('strict_hist60'),
            pl.col('formation_member_bp_p20').alias('bp_p20'),
            pl.col('formation_member_bp_p50').alias('bp_p50'),
            pl.col('formation_member_bp_p90').alias('bp_p90'),
            pl.col('formation_member_n_nyse_snapshot').alias('n_nyse_snapshot'),
        ])
        .collect(engine='streaming')
    )

memberships = {name: extract_formation_membership(frame) for name, frame in datasets.items()}

for name, membership in memberships.items():
    assert membership.select(['formation_month', 'PERMNO']).is_duplicated().sum() == 0
    assert membership.filter(
        pl.col('active_month') != pl.col('formation_month').dt.offset_by('1mo')
    ).is_empty(), f'{name}: incorrect M to M+1 lag'
    assert membership.filter(pl.col('DlyCalDt').dt.truncate('1mo') != pl.col('formation_month')).is_empty()
    assert membership.filter(~pl.col('strict_hist60')).is_empty()

    counts = membership.group_by('formation_month').agg(
        pl.len().alias('n'),
        pl.col('rank').n_unique().alias('n_ranks'),
        pl.col('rank').min().alias('min_rank'),
        pl.col('rank').max().alias('max_rank'),
    ).sort('formation_month')
    failures = counts.filter(
        (pl.col('n') != TOP_N) | (pl.col('n_ranks') != TOP_N)
        | (pl.col('min_rank') != 1) | (pl.col('max_rank') != TOP_N)
    )
    assert failures.is_empty(), f'{name}: incomplete formation memberships\n{failures}'
    print(name, f': {membership["formation_month"].n_unique()} formation months, exactly {TOP_N} assets each')

## 4. Independent NYSE breakpoint reconstruction

The reference population is reconstructed from the raw source at each market close. Only alive NYSE securities enter P20, P50, and P90. This test verifies both the breakpoint values and the recorded NYSE snapshot size.

In [ ]:
source_us = (
    pl.scan_parquet(SRC)
    .filter(pl.col('is_formation_eligible'))
    .select(['PERMNO', 'DlyCalDt', 'DlyCap', 'DlyPrc', 'PrimaryExch', 'DelistingDt'])
    .unique(subset=['PERMNO', 'DlyCalDt'], keep='first')
    .with_columns(pl.col('DlyCalDt').dt.truncate('1mo').alias('month'))
)
market_end = (
    source_us.select(['month', 'DlyCalDt']).unique()
    .group_by('month').agg(pl.col('DlyCalDt').max().alias('market_end'))
)
monthly_snapshot = (
    source_us.sort(['PERMNO', 'DlyCalDt'])
    .group_by(['PERMNO', 'month'])
    .agg(
        pl.col('DlyCalDt').last().alias('security_last_date'),
        pl.col('PrimaryExch').last().alias('formation_exchange'),
        pl.col('DlyCap').last().alias('mktcap'),
        pl.col('DlyPrc').last().alias('month_end_price'),
        pl.col('DelistingDt').drop_nulls().last().alias('DelistingDt'),
    )
    .join(market_end, on='month', how='left')
    .filter(
        pl.col('security_last_date') == pl.col('market_end'),
        (pl.col('DelistingDt').is_null()) | (pl.col('DelistingDt') > pl.col('market_end')),
        pl.col('mktcap').is_not_null(),
        pl.col('mktcap') > 0,
        pl.col('month_end_price').is_not_null(),
        pl.col('formation_exchange') == NYSE_CODE,
    )
)
independent_breakpoints = (
    monthly_snapshot.group_by('month')
    .agg(
        pl.col('mktcap').quantile(0.20).alias('bp_p20_check'),
        pl.col('mktcap').quantile(0.50).alias('bp_p50_check'),
        pl.col('mktcap').quantile(0.90).alias('bp_p90_check'),
        pl.len().alias('n_nyse_check'),
    )
    .collect(engine='streaming')
)

recorded_breakpoints = pl.concat([
    membership.select(['formation_month', 'bp_p20', 'bp_p50', 'bp_p90', 'n_nyse_snapshot'])
              .unique('formation_month')
    for membership in memberships.values()
]).unique('formation_month')
check = recorded_breakpoints.join(
    independent_breakpoints, left_on='formation_month', right_on='month', how='left'
)
for p in ('20', '50', '90'):
    assert check.filter((pl.col(f'bp_p{p}') - pl.col(f'bp_p{p}_check')).abs() > 1e-10).is_empty()
assert check.filter(pl.col('n_nyse_snapshot') != pl.col('n_nyse_check')).is_empty()
print('NYSE breakpoint values and snapshot sizes: PASS.')

## 5. Bucket and exchange contracts

In [ ]:
big_m = memberships['big_caps']
small_m = memberships['small_caps_p20_p50']

assert big_m.filter(pl.col('bucket') != 'big_caps').is_empty()
assert small_m.filter(pl.col('bucket') != 'small_caps_p20_p50').is_empty()
assert big_m.filter(~pl.col('formation_exchange').is_in(INVESTABLE_EXCHANGES)).is_empty()
assert big_m.filter(pl.col('formation_exchange') != NYSE_CODE).height > 0, (
    'The big-cap universe contains no AMEX/NASDAQ formation member; the NYSE P90 may have been applied to NYSE only.'
)
assert small_m.filter(pl.col('formation_exchange') != NYSE_CODE).is_empty()
assert big_m.filter(pl.col('formation_mktcap') <= pl.col('bp_p90')).is_empty()
assert small_m.filter(
    (pl.col('formation_mktcap') <= pl.col('bp_p20'))
    | (pl.col('formation_mktcap') > pl.col('bp_p50'))
).is_empty()

overlap = big_m.select(['formation_month', 'PERMNO']).join(
    small_m.select(['formation_month', 'PERMNO']),
    on=['formation_month', 'PERMNO'], how='inner',
)
assert overlap.is_empty(), f'{len(overlap)} memberships overlap across buckets'
print('Bucket definitions and exchange scopes: PASS.')
display(big_m.group_by('formation_exchange').agg(pl.len().alias('memberships')).sort('formation_exchange'))

## 6. Delisting information is exposed only on the event date

In [ ]:
for name, frame in datasets.items():
    stats = frame.select(
        pl.col('is_delisting_event').sum().alias('events'),
        (pl.col('DelRet_event').is_not_null() & ~pl.col('is_delisting_event')).sum().alias('future_leaks'),
        (pl.col('is_delisting_event') & pl.col('DlyRet').is_null()).sum().alias('missing_event_return'),
    ).collect(engine='streaming')
    assert stats['future_leaks'][0] == 0, f'{name}: future delisting information leaked'
    assert stats['missing_event_return'][0] == 0, f'{name}: missing delisting-event return'
    print(name)
    display(stats)

## 7. Actual 60-month completeness

Completeness is independently reconstructed from the exported daily histories against the common NYSE–AMEX–NASDAQ calendar. Every formation member must have 60 consecutive complete calendar months ending at formation.

In [ ]:
calendar = (
    pl.scan_parquet(SRC)
    .filter(pl.col('is_formation_eligible'))
    .select('DlyCalDt').unique()
    .with_columns(pl.col('DlyCalDt').dt.truncate('1mo').alias('month'))
    .group_by('month').agg(pl.col('DlyCalDt').n_unique().alias('n_market_days'))
    .collect(engine='streaming')
)

def actual_strict_history(frame: pl.LazyFrame) -> pl.DataFrame:
    return (
        frame.select(['PERMNO', 'DlyCalDt', 'DlyRet'])
        .with_columns(pl.col('DlyCalDt').dt.truncate('1mo').alias('month'))
        .group_by(['PERMNO', 'month'])
        .agg(
            pl.col('DlyCalDt').n_unique().alias('n_security_days'),
            pl.col('DlyRet').is_not_null().sum().alias('n_valid_returns'),
        )
        .collect(engine='streaming')
        .join(calendar, on='month', how='left')
        .with_columns(
            (
                (pl.col('n_security_days') == pl.col('n_market_days'))
                & (pl.col('n_valid_returns') == pl.col('n_market_days'))
            ).alias('complete_month'),
            (pl.col('month').dt.year() * 12 + pl.col('month').dt.month()).alias('month_id'),
        )
        .sort(['PERMNO', 'month_id'])
        .with_columns(
            pl.col('month_id').shift(W_MAX - 1).over('PERMNO').alias('month_id_lag59'),
            pl.col('complete_month').cast(pl.Int16)
              .rolling_sum(window_size=W_MAX, min_samples=W_MAX)
              .over('PERMNO').alias('n_complete_last60'),
        )
        .with_columns(
            (
                pl.col('month_id_lag59').is_not_null()
                & ((pl.col('month_id') - pl.col('month_id_lag59')) == W_MAX - 1)
                & (pl.col('n_complete_last60') == W_MAX)
            ).alias('actual_strict_hist60')
        )
    )

for name, frame in datasets.items():
    actual = actual_strict_history(frame)
    checked = memberships[name].join(
        actual.select(['PERMNO', 'month', 'actual_strict_hist60']),
        left_on=['PERMNO', 'formation_month'], right_on=['PERMNO', 'month'], how='left',
    )
    failures = checked.filter(~pl.col('actual_strict_hist60').fill_null(False))
    assert failures.is_empty(), f'{name}: {len(failures)} memberships lack a full 60-month matrix'
    print(name, ': every formation has a complete 60-month daily matrix.')

## 8. Active-month availability is not formation membership

A security may disappear after selection. Such a case may reduce observed `is_active` membership in $M+1$, but it must leave the 100-name formation membership unchanged. This diagnostic reports those cases without using them as an eligibility filter.

In [ ]:
for name, frame in datasets.items():
    observed_active = (
        frame.filter(pl.col('is_active') == 1)
        .select(['formation_month', 'PERMNO']).unique()
        .group_by('formation_month').agg(pl.len().alias('n_observed_m_plus_1'))
        .collect(engine='streaming')
    )
    formation_counts = (
        memberships[name].group_by('formation_month')
        .agg(pl.len().alias('n_formation'))
    )
    diagnostic = formation_counts.join(observed_active, on='formation_month', how='left').with_columns(
        pl.col('n_observed_m_plus_1').fill_null(0),
        (pl.col('n_formation') - pl.col('n_observed_m_plus_1').fill_null(0)).alias('post_formation_disappearances'),
    )
    assert diagnostic['n_formation'].eq(TOP_N).all()
    print(name, ': formation membership remains fixed at 100')
    display(diagnostic.filter(pl.col('post_formation_disappearances') > 0).sort('formation_month'))

## 9. Holding paths survive exchange-code migrations

Exchange eligibility is a formation-time condition. Once selected, the same `PERMNO` must retain every available CRSP observation through $M+1$, even if `PrimaryExch` subsequently leaves the N/A/Q formation set. A surviving holding leg must therefore cover the complete market calendar; only a documented delisting may end earlier.

In [ ]:
holding_calendar = (
    pl.scan_parquet(SRC)
    .filter(pl.col('is_formation_eligible'))
    .select('DlyCalDt').unique()
    .with_columns(pl.col('DlyCalDt').dt.truncate('1mo').alias('active_month'))
    .group_by('active_month')
    .agg(pl.col('DlyCalDt').n_unique().alias('n_market_days'))
)

for name, frame in datasets.items():
    legs = (
        frame.filter(pl.col('is_active') == 1)
        .group_by(['formation_month', 'active_month', 'PERMNO'])
        .agg(
            pl.col('DlyCalDt').n_unique().alias('n_observed_days'),
            pl.col('DlyCalDt').max().alias('last_observed_date'),
            pl.col('DlyCalDt').filter(pl.col('is_delisting_event')).max().alias('event_date'),
            pl.col('is_delisting_event').any().alias('delisted'),
            (~pl.col('PrimaryExch').is_in(INVESTABLE_EXCHANGES)).any().alias('migrated_outside_formation_set'),
        )
        .join(holding_calendar, on='active_month', how='left')
        .collect(engine='streaming')
    )
    incomplete_survivors = legs.filter(
        ~pl.col('delisted') & (pl.col('n_observed_days') != pl.col('n_market_days'))
    )
    post_event_rows = legs.filter(
        pl.col('delisted') & (pl.col('last_observed_date') != pl.col('event_date'))
    )
    assert incomplete_survivors.is_empty(), (
        f'{name}: a non-delisted holding path is incomplete\n{incomplete_survivors}'
    )
    assert post_event_rows.is_empty(), (
        f'{name}: observations remain after a delisting event\n{post_event_rows}'
    )
    n_migrations = legs.filter(pl.col('migrated_outside_formation_set')).height
    print(name, f': complete M+1 paths; {n_migrations} holding legs cross outside N/A/Q without truncation')


In [ ]:
# 10. CONCLUSION
print('=' * 78)
print('ALL ASSERTIONS PASSED')
print('NYSE breakpoints are correct; big caps cover N/A/Q; P20–P50 is NYSE-only.')
print('Every formation month contains exactly 100 ex-ante members with a full 60-month history.')
print('The same memberships can therefore support 36/48/60-month signal windows.')
print('Holding-period paths remain complete across post-formation exchange migrations.')
print('=' * 78)